# Banking Marketing Analytics & Conversational BI Assistant

## Project Overview

This project analyzes banking marketing campaign data using SQL, statistics, machine learning, and conversational analytics techniques in Databricks.

### Objectives
- Perform exploratory data analysis (EDA)
- Validate business relationships using statistical testing
- Build a machine learning model to predict customer subscription behavior
- Develop a prototype conversational Text-to-SQL analytics assistant

### Tools & Technologies
- Databricks
- SQL
- Python
- Pandas
- SciPy
- Scikit-learn
- Spark SQL


In [0]:
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
from sklearn.model_selection import train_test_split


In [0]:
table = [
    [3386, 4940],
    [307, 230],
    [618, 610],
    [978, 93]
]
print(table)

[[3386, 4940], [307, 230], [618, 610], [978, 93]]


# Statistical Validation

Statistical hypothesis testing was performed to validate relationships between customer attributes and term deposit subscription behavior.

### Statistical Methods Used
- Chi-Square Test of Independence
- Welch’s Two-Sample t-test

### Goals
- Determine whether previous campaign outcome is associated with subscription behavior
- Compare customer balance differences between subscribed and non-subscribed customers

In [0]:
chi2, p_value, dof, expected = chi2_contingency(table)

In [0]:
print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)
print("Expected Frequencies:")
print(expected)

Chi-square statistic: 1004.635780185333
P-value: 1.7761850102620285e-217
Degrees of freedom: 3
Expected Frequencies:
[[3945.19028848 4380.80971152]
 [ 254.45197993  282.54802007]
 [ 581.87529117  646.12470883]
 [ 507.48244042  563.51755958]]


In [0]:
df = spark.table("demo.default.bank").toPandas()
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


T-Test

In [0]:
balance_yes = df[df["deposit"] == "yes"]["balance"]
balance_no = df[df["deposit"] == "no"]["balance"]

In [0]:
t_stat, p_value = ttest_ind(balance_yes, balance_no, equal_var=False)

In [0]:
print("T-statistic:", t_stat)
print("P-value:", p_value)
print("Average balance - subscribed:", balance_yes.mean())
print("Average balance - not subscribed:", balance_no.mean())

T-statistic: 8.520420767660433
P-value: 1.8105789895875916e-17
Average balance - subscribed: 1804.2679145396105
Average balance - not subscribed: 1280.2271411544355


In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB


In [0]:
df["deposit"].value_counts()

deposit
no     5873
yes    5289
Name: count, dtype: int64

In [0]:
df[["poutcome", "balance", "deposit"]].head()

,poutcome,balance,deposit
0,unknown,2343,yes
1,unknown,45,yes
2,unknown,1270,yes
3,unknown,2476,yes
4,unknown,184,yes


In [0]:
df.groupby("deposit")["balance"].mean()

deposit
no     1280.227141
yes    1804.267915
Name: balance, dtype: float64

In [0]:
df.groupby("poutcome")["deposit"].value_counts()

poutcome  deposit
failure   yes         618
          no          610
other     yes         307
          no          230
success   yes         978
          no           93
unknown   no         4940
          yes        3386
Name: count, dtype: int64

# Machine Learning

A Logistic Regression model was developed to predict whether a customer subscribes to a term deposit based on selected customer and campaign features.

### Machine Learning Workflow
- Feature selection
- One-hot encoding
- Train-test split
- Logistic Regression model training
- Model evaluation using accuracy, confusion matrix, precision, recall, and F1-score

In [0]:
ml_df = df.copy()

In [0]:
ml_df = ml_df[["balance", "poutcome", "deposit"]]
ml_df.head()

,balance,poutcome,deposit
0,2343,unknown,yes
1,45,unknown,yes
2,1270,unknown,yes
3,2476,unknown,yes
4,184,unknown,yes


In [0]:
mapping = {"yes":1, "no":0}
ml_df["deposit"] = ml_df["deposit"].map(mapping)

In [0]:
ml_df.head()

,balance,poutcome,deposit
0,2343,unknown,1
1,45,unknown,1
2,1270,unknown,1
3,2476,unknown,1
4,184,unknown,1


In [0]:
ml_df = pd.get_dummies(ml_df, columns=["poutcome"], drop_first=True)

In [0]:
ml_df.head(20)

,balance,deposit,poutcome_other,poutcome_success,poutcome_unknown
0,2343,1,False,False,True
1,45,1,False,False,True
2,1270,1,False,False,True
3,2476,1,False,False,True
4,184,1,False,False,True
5,0,1,False,False,True
6,830,1,False,False,True
7,545,1,False,False,True
8,1,1,False,False,True
9,5090,1,False,False,True


In [0]:
ml_df[ml_df["poutcome_other"] == True]

,balance,deposit,poutcome_other,poutcome_success,poutcome_unknown
890,-247,1,True,False,False
961,1494,1,True,False,False
968,0,1,True,False,False
977,1429,1,True,False,False
982,149,1,True,False,False
...,...,...,...,...,...
11054,162,0,True,False,False
11073,6718,0,True,False,False
11080,199,0,True,False,False
11104,971,0,True,False,False


In [0]:
ml_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   balance           11162 non-null  int64
 1   deposit           11162 non-null  int64
 2   poutcome_other    11162 non-null  bool 
 3   poutcome_success  11162 non-null  bool 
 4   poutcome_unknown  11162 non-null  bool 
dtypes: bool(3), int64(2)
memory usage: 207.2 KB


In [0]:
X = ml_df.drop("deposit", axis=1)
y = ml_df["deposit"]

In [0]:
print(X.shape)
print(y.shape)

(11162, 4)
(11162,)


In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [0]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(8929, 4)
(2233, 4)
(8929,)
(2233,)


In [0]:
from sklearn.linear_model import LogisticRegression

In [0]:
model = LogisticRegression()

In [0]:
model.fit(X_train, y_train)

/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [0]:
y_pred = model.predict(X_test)

In [0]:
print(y_pred[:10])

[0 0 1 0 0 0 0 0 0 1]


In [0]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [0]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.6144200626959248


In [0]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[959 207]
 [654 413]]


In [0]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.59      0.82      0.69      1166
           1       0.67      0.39      0.49      1067

    accuracy                           0.61      2233
   macro avg       0.63      0.60      0.59      2233
weighted avg       0.63      0.61      0.59      2233



# Conversational Analytics Assistant

This section develops a lightweight rule-based conversational analytics assistant using Python and Pandas.

The assistant accepts simple natural-language business questions and maps them to predefined analytical queries.

This is not a true LLM or GenAI model. Instead, it is a beginner-level prototype that demonstrates the workflow behind conversational business intelligence tools: user question -> intent detection -> data query -> analytical result.

In [0]:
def mini_bank_ai_assistant(prompt, df):
    prompt = prompt.lower()

    # Intent 1: Which jobs subscribe the most?
    if (
        ("job" in prompt or "jobs" in prompt or "occupation" in prompt)
        and ("subscribe" in prompt or "deposit" in prompt)
        and ("most" in prompt or "top" in prompt or "highest" in prompt or "best" in prompt)
    ):
        result = (
            df.groupby("job")["deposit"]
            .apply(lambda x: (x == "yes").mean() * 100)
            .reset_index(name="subscription_rate")
            .sort_values("subscription_rate", ascending=False)
        )
        return result

    # Intent 2: Count subscribed customers
    elif (
        ("how many" in prompt or "number" in prompt or "count" in prompt)
        and ("subscribe" in prompt or "deposit" in prompt or "yes" in prompt)
    ):
        result = df[df["deposit"] == "yes"].shape[0]
        return f"Number of subscribed customers: {result}"

    # Intent 3: Average balance by deposit
    elif (
        "average balance" in prompt
        and ("deposit" in prompt or "subscription" in prompt)
    ):
        result = (
            df.groupby("deposit")["balance"]
            .mean()
            .reset_index(name="average_balance")
        )
        return result

    else:
        return "Sorry, I do not understand this question yet."

In [0]:
mini_bank_ai_assistant("Which jobs subscribe the most?", df)

,job,subscription_rate
8,student,74.722222
5,retired,66.323907
10,unemployed,56.582633
4,management,50.701481
11,unknown,48.571429
0,admin.,47.301349
6,self-employed,46.172840
9,technician,46.077894
7,services,39.978332
3,housemaid,39.781022


In [0]:
mini_bank_ai_assistant("Show me number of customers where deposit is yes", df)

'Number of subscribed customers: 5289'

In [0]:
mini_bank_ai_assistant("Show average balance by deposit", df)

,deposit,average_balance
0,no,1280.227141
1,yes,1804.267915


## Conversational Analytics Assistant Summary

The assistant successfully maps natural-language questions to analytical outputs using rule-based intent detection.

Example:
- User question: "Which jobs subscribe the most?"
- Detected intent: subscription rate by job
- Output: ranked table of job categories by subscription rate

This prototype demonstrates the basic logic behind conversational analytics systems and can later be extended using an actual LLM API to generate SQL dynamically.

# Conversational Text-to-SQL Analytics Assistant

A lightweight prototype conversational analytics assistant was developed using Python and Spark SQL.

The assistant accepts natural-language business questions and dynamically generates SQL queries to retrieve analytical insights from the banking dataset.

### Example Queries
- "Show number of customers where deposit is yes"
- "How many customers are not subscribed?"

### Project Goal
This prototype demonstrates beginner-level conversational analytics and AI-inspired business intelligence workflows without using external LLM APIs.

In [0]:
def text_to_sql_assistant(prompt):
    prompt = prompt.lower()

    if (
        ("how many" in prompt or "number" in prompt or "count" in prompt)
        and ("deposit is yes" in prompt or "subscribed" in prompt)
    ):
        sql_query = """
        SELECT COUNT(*) AS subscribed_customers
        FROM demo.default.bank
        WHERE deposit = 'yes'
        """

    elif (
        ("how many" in prompt or "number" in prompt or "count" in prompt)
        and ("deposit is no" in prompt or "not subscribed" in prompt)
    ):
        sql_query = """
        SELECT COUNT(*) AS non_subscribed_customers
        FROM demo.default.bank
        WHERE deposit = 'no'
        """

    else:
        return "Sorry, I do not understand this question yet."

    print("Generated SQL Query:")
    #print(sql_query)

    result = spark.sql(sql_query)
    display(result)

In [0]:
text_to_sql_assistant("Show number of customers where deposit is yes")

Generated SQL Query:


subscribed_customers
5289


In [0]:
text_to_sql_assistant("How many customers are not subscribed?")

Generated SQL Query:


subscribed_customers
5289


# Final Findings

### Key Insights
- Previous campaign outcome (`poutcome`) showed a strong relationship with customer subscription behavior.
- Customers with previous successful campaign outcomes had the highest subscription rates.
- Subscribed customers maintained higher average balances than non-subscribed customers.
- Logistic Regression achieved moderate predictive performance using a limited feature set.

### Project Outcomes
This project demonstrates an end-to-end data science workflow integrating:
- SQL analytics
- Statistical hypothesis testing
- Machine learning
- Conversational Text-to-SQL analytics

### Future Improvements
- Add more predictive features
- Improve machine learning performance
- Integrate real LLM APIs for semantic SQL generation
- Develop advanced conversational analytics workflows